# TEM Analyzer v2.3.0 — 회사 GPU 서버 setup
ABL 1.4처럼 **기존 커널/로컬 가중치를 재사용**하는 설치·점검·실행 노트북입니다. 학습/데이터 분할/GT 변환은 하지 않습니다.

- `TEM_Analyzer_v2` **전체 소스 폴더**를 서버에 올리고 이 파일을 엽니다. 노트북만으로는 실행되지 않습니다.
- Python 3.11 이상 / 64-bit. 기존 ABL CUDA 커널을 사용할 수 있지만 **학습 중 패키지 설치는 금지**하고, 가능하면 별도 복제 환경을 사용하세요.
- 아래 설치/서버 시작/종료는 각각 명시적 스위치가 있습니다. **Run All만으로 설치·서버 종료를 하지 않습니다.**
- 기본값은 NVIDIA CUDA 필수입니다. CPU 확인만 할 때 `REQUIRE_CUDA=False`로 명시하세요.
- 회사 영상/학습 파일/토큰을 노트북 출력에 첨부하거나 GitHub에 올리지 마세요.
- 세부 안내: [docs/INSTALL_KO.md](docs/INSTALL_KO.md). 회사 서버에서의 실제 성공 여부는 아래 셀 결과로 확인합니다.


## 1. 경로·환경 설정
프로젝트 루트에서 열었다면 `APP_DIR=""` 그대로 실행합니다. 경로를 찾지 못하면 압축을 푼 **TEM_Analyzer_v2 전체 경로**를 입력하세요.

아래 `USE_OCR=False`는 **설치/검증에서 OCR을 생략**하는 선택이며 UI에서 OCR 버튼 자체를 숨기지는 않습니다.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json

APP_DIR = ""              # 예: "/workspace/TEM_Analyzer_v2"; 비어 있으면 현재/상위 폴더 탐색
REQUIRE_CUDA = True        # GPU가 없으면 중단. CPU 시험은 False
USE_OCR = True             # OCR 가중치 2개가 없으면 False로 SAM 먼저 확인 가능

if APP_DIR:
    ROOT = Path(APP_DIR).expanduser().resolve()
else:
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    ROOT = next((p for p in candidates if (p / "tools/server_setup.py").is_file()), None)
    if ROOT is None:
        raise FileNotFoundError("APP_DIR에 TEM_Analyzer_v2 전체 경로를 입력하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from tools.server_setup import app_root, install_packages, model_paths, NotebookServer
ROOT = app_root(ROOT)
print("앱:", ROOT)
print("노트북 Python:", sys.executable)
print("Python:", sys.version.split()[0])
print("GPU 필수:", REQUIRE_CUDA)


## 2. 최초 한 번 패키지 설치 (다음 실행부터 생략)
`INSTALL_PACKAGES=True`로 변경해야 실제 설치합니다. 기존 torch/torchvision 및 설치된 주요 수치 패키지는 버전을 고정합니다. 충돌을 해결하려고 기존 CUDA를 CPU로 바꾸거나 강제 업그레이드하지 않습니다.

- 사내 pip 설정이 있으면 `PACKAGE_INDEX=""`를 유지합니다. URL에 비밀번호/토큰을 넣지 마세요.
- 완전 오프라인이면 **서버와 같은 OS/Python/CUDA**용 wheel을 `wheelhouse/`에 넣고 `WHEELHOUSE=str(ROOT/"wheelhouse")`로 설정합니다. Windows wheel을 Linux에 복사하면 안 됩니다.
- torch 쌍이 이미 있으면 `TORCH_PACKAGES=[]` 그대로 둡니다. 둘 다 없을 때만 IT가 승인한 **정확한 버전 2개**와 인덱스를 지정합니다. 예시 문자열의 "버전"은 실제 숫자로 바꿔야 합니다.
- 충돌 시 오류를 확인하고 별도 환경을 준비하세요. 설치된 CUDA 패키지가 있다는 사실만으로 GPU 동작이 보장되지는 않습니다.


In [ ]:
INSTALL_PACKAGES = False  # 최초 패키지 보충 시에만 True
PACKAGE_INDEX = ""         # 자격 증명 없는 사내 HTTPS 패키지 미러 (선택)
WHEELHOUSE = ""            # 지정 시 --no-index로만 설치
TORCH_INDEX = ""           # 신규 torch 설치일 때만 승인된 CUDA wheel 인덱스
TORCH_PACKAGES = []        # 둘 다 없을 때만 ["torch==버전", "torchvision==버전"]

install_packages(
    ROOT, enabled=INSTALL_PACKAGES, use_ocr=USE_OCR,
    wheelhouse=WHEELHOUSE, package_index=PACKAGE_INDEX,
    torch_index=TORCH_INDEX, torch_packages=TORCH_PACKAGES,
)


## 3. 실제 환경 점검
같은 Python의 **새 프로세스**에서 import, pip 충돌, CPU 및 CUDA의 tensor/NMS 연산을 검사합니다. 그러므로 설치 전 노트북에 남은 torch import 캐시로 결과를 판단하지 않습니다.

CUDA 검사가 실패하면 GPU 할당, 서버 드라이버, 선택한 커널, PyTorch 빌드를 확인하세요. 노트북은 드라이버를 설치하거나 권한을 변경하지 않습니다. 기존 ABL이 Python 3.10 환경이었다면 Analyzer용 Python 3.11 이상 환경이 필요합니다.


In [ ]:
env_cmd = [sys.executable, "-u", str(ROOT / "tools/server_setup.py"),
           "environment", "--root", str(ROOT)]
if not REQUIRE_CUDA:
    env_cmd.append("--cpu")
if not USE_OCR:
    env_cmd.append("--no-ocr")
subprocess.run(env_cmd, cwd=ROOT, check=True)


## 4. ABL 1.4 파일을 어디에 넣나요?
```text
TEM_Analyzer_v2/
├── setup.ipynb
├── run.py
├── tools/server_setup.py
├── tem_analyzer/                 ← Analyzer 소스, ABL 소스로 덮어쓰지 않음
├── web/
├── adaptation_backend/          ← SAM/Refiner 소스 이미 포함
├── models/
│   ├── sam_vit_h_4b8939.pth      ← ABL의 weights/에서 복사 (필수)
│   ├── abl14/
│   │   └── decoder_abl/
│   │       ├── adaptation.pt    ← ABL export 결과 (선택)
│   │       ├── inference_settings.json  ← 기록용, 선택
│   │       └── model_manifest.json      ← 기록용, 선택
│   └── easyocr/
│       ├── craft_mlt_25k.pth    ← OCR 사용 시 필수, ABL에는 없음
│       └── english_g2.pth      ← OCR 사용 시 필수, ABL에는 없음
└── projects/
    └── company_session/        ← Analyzer가 저장하는 작업 프로젝트
```

- **기본 SAM만 실행**하면 `ADAPTATION=""`. ABL 학습 결과까지 쓰려면 export한 `adaptation.pt` 경로를 지정합니다. Decoder/Refiner 실험은 폴더를 나누고 한 번에 하나를 선택합니다.
- `02_evaluate_export.ipynb`의 실제 출력 경로에서 가져옵니다. `best.pt`도 schema=1 bundle이면 로드할 수 있지만 export 파일이 우선입니다.
- 기본 SAM은 **학습에 사용한 바로 그 파일**이어야 합니다. 모델 종류/SHA256/전처리/state 호환성은 다음 실제 로드에서 검사합니다. 학습 결과 하나만 복사해서는 안 됩니다.
- ABL의 `.venv`, `temlab`, `vendor`, `requirements.txt`, config, 학습 데이터, embedding cache는 옮기지 않아도 됩니다.
- 이미 서버에 파일이 있으면 아래 경로에 절대 경로를 지정해 공간을 절약할 수 있습니다. 자동 복사/덮어쓰기/다운로드는 하지 않습니다.
- OCR 두 파일은 이전 Analyzer에서 복사하거나 [검출 ZIP](https://github.com/JaidedAI/EasyOCR/releases/download/pre-v1.1.6/craft_mlt_25k.zip), [영어 ZIP](https://github.com/JaidedAI/EasyOCR/releases/download/v1.3/english_g2.zip)을 승인된 방법으로 받고 **압축을 풀어** 배치합니다.


In [ ]:
SAM_VARIANT = "vit_h"  # ABL 기본 모델. B를 썼다면 vit_b 및 파일명도 변경
SAM_CHECKPOINT = "models/sam_vit_h_4b8939.pth"
ADAPTATION = ""        # 선택: "models/abl14/decoder_abl/adaptation.pt"
OCR_DIR = "models/easyocr"
PROJECT_DIR = "projects/company_session"  # 기존 프로젝트는 전체 폴더의 절대 경로
PORT = 8765

FILES = model_paths(ROOT, SAM_VARIANT, SAM_CHECKPOINT, ADAPTATION, OCR_DIR, USE_OCR)
DEVICE = "cuda" if REQUIRE_CUDA else "cpu"
print(json.dumps(FILES, ensure_ascii=False, indent=2))
print("사용 장치:", DEVICE)
print("프로젝트:", PROJECT_DIR)


## 5. 실제 SAM/선택적 OCR 시험 — 회사 영상 없이 검사
로컬 가중치를 읽고 합성 이미지에서 두 번 추론합니다. 첫 추론은 이미지 인코딩 포함, 두 번째는 임베딩 재사용입니다. CUDA 시간은 동기화해서 측정합니다. 검사가 끝나면 별도 프로세스가 종료되어 **테스트 모델의 GPU 메모리를 반환**합니다.

이 시험은 실행/형식 검사이며 실제 TEM 경계 정확도 검증은 아닙니다. 학습 bundle의 mask가 합성 이미지에서 비어도 실행 검사와 회사 정확도를 구분해야 합니다. ABL 학습 시 AMP 설정과 Analyzer 추론의 정밀도 차이도 있어 수치가 완전히 같다고 단정하지 않습니다. OOM이면 다른 학습 프로세스의 메모리 사용을 확인하고 작업을 조정하세요. 다른 프로세스는 자동 종료하지 않습니다.


In [ ]:
RUN_SAM_TEST = True  # 첫 확인 후 다음 날 실행부터 False로 생략 가능
if RUN_SAM_TEST:
    smoke_cmd = [sys.executable, "-u", str(ROOT / "tools/server_setup.py"),
                 "smoke", "--root", str(ROOT),
                 "--checkpoint", FILES["checkpoint"], "--variant", SAM_VARIANT]
    if DEVICE == "cpu":
        smoke_cmd.append("--cpu")
    if FILES["adaptation_path"]:
        smoke_cmd += ["--adaptation", FILES["adaptation_path"]]
    if USE_OCR:
        smoke_cmd += ["--ocr-dir", FILES["ocr_dir"]]
    subprocess.run(smoke_cmd, cwd=ROOT, check=True)
else:
    print("SAM 시험 생략")


## 6. 웹 서버 시작
`START_SERVER=True`로 바꾸고 실행합니다. 모델을 서버에서 자동 로드하려면 `LOAD_MODEL_ON_START=True`를 유지하세요. 첫 로드 시 가중치 해시 검사와 GPU 메모리 할당으로 시간이 걸릴 수 있습니다.

- 이미 이 노트북에서 실행한 서버는 재사용합니다. 다른 프로세스가 포트를 사용하면 멈추고 안내합니다.
- 프로젝트당 서버 1개만 사용하세요. 동일 프로젝트 동시 편집은 지원하지 않습니다.
- `127.0.0.1`만 바인딩하며 **인증 없는 공개 서비스로 노출하지 않습니다**.
- UI에서 모델을 직접 바꿀 때는 아래 출력한 경로/variant/device를 사용하세요. OCR 폴더는 UI의 **OCR 설정 → 로컬 EasyOCR 폴더**에 입력합니다.


In [ ]:
START_SERVER = False       # 실행할 때 True
LOAD_MODEL_ON_START = True

if START_SERVER:
    wanted = NotebookServer(ROOT, PROJECT_DIR, PORT)
    previous = globals().get("TEM_SERVER")
    if previous is not None and previous.process is not None and previous.process.poll() is None:
        if (previous.root, previous.project, previous.port) != (wanted.root, wanted.project, wanted.port):
            raise RuntimeError("기존 서버가 실행 중입니다. 작업 저장 후 종료 셀을 실행하고 설정을 변경하세요.")
        TEM_SERVER = previous
    else:
        TEM_SERVER = wanted
    TEM_SERVER.start()
    payload = {k: FILES[k] for k in ("checkpoint", "variant", "adaptation_path")}
    payload["device"] = DEVICE
    if LOAD_MODEL_ON_START:
        current_model = TEM_SERVER.request("/api/state").get("model") or {}
        if any(current_model.get(k) != v for k, v in payload.items()):
            # A large ViT-H/model hash check can take time; request goes only to our loopback process.
            print(TEM_SERVER.request("/api/model/load", payload, timeout=900))
        else:
            print("같은 모델이 이미 로드되어 있습니다. 파일 자체를 교체했다면 UI에서 다시 로드하세요.")
    print("서버 주소:", TEM_SERVER.url)
    print("UI OCR 폴더:", FILES["ocr_dir"] if USE_OCR else "OCR 검사 생략 상태")
    print("아래 원격 접속 안내를 확인하세요.")
else:
    print("서버 미시작. START_SERVER=True로 바꿔 이 셀을 실행하세요.")


## 7. 내 PC에서 계측 UI 열기
서버의 localhost 주소는 PC의 localhost와 다릅니다. **회사에서 허용하는 포트 전달**을 사용하세요.

**SSH가 가능한 경우 — 아래 명령은 노트북이 아니라 내 PC 터미널에서 실행**
```text
ssh -N -L 8765:127.0.0.1:8765 USER@SERVER
```
계정/서버/포트는 실제 값으로 바꿉니다. 연결을 유지한 채 PC 브라우저에서 `http://127.0.0.1:8765/`를 엽니다. VS Code Remote-SSH의 Ports 탭에서 8765를 **Private**로 전달해도 됩니다.

**중요한 제약**
- 노트북 커널이 SSH 서버와 다른 계산 노드/컨테이너이면 위 명령만으로 연결되지 않습니다. IT가 허용한 노드 전달 경로가 필요합니다.
- 현재 앱의 `/api`·`/web`는 루트 절대 경로입니다. **일반 JupyterHub `/user/.../proxy/8765/` 링크만 붙이는 방식은 지원 완료 상태가 아닙니다.** SSH 없이 프록시만 가능한 서버라면 해당 URL 구조에 맞춘 추가 대응이 필요합니다.
- Jupyter 프록시를 설치할 때는 커널 환경이 아니라 Jupyter 서버 환경에 설치/활성화해야 합니다. 이 노트북은 프록시·방화벽·인증 설정을 변경하지 않습니다.
- 상시 서비스/다중 사용자 운영은 별도 설계가 필요합니다. Jupyter 세션/작업 스케줄러가 종료되면 웹 서버도 종료될 수 있습니다.


## 8. 로그 확인 (문제가 있을 때)
회사 경로/오류 정보가 포함될 수 있으므로 이 출력 전체를 공개 저장소에 업로드하지 마세요.


In [ ]:
server = globals().get("TEM_SERVER")
if server is not None and server.log_path is not None:
    print("PID:", server.process.pid, "exit code:", server.process.poll())
    print(server.log_path.read_text(encoding="utf8", errors="replace")[-6000:])
else:
    print("이 커널에서 시작한 서버가 없습니다.")


## 9. 작업 완료 후 종료
UI의 SAM/일괄/저장 작업이 모두 끝난 것을 확인한 후 `STOP_SERVER=True`로 바꿔 실행합니다. **Run All은 기본값 False이므로 방금 시작한 서버를 바로 종료하지 않습니다.** Windows에서는 terminate가 강제 종료이므로 저장 중 실행하면 안 됩니다.

이 노트북이 시작한 프로세스만 종료하며 프로젝트/모델 파일은 삭제하지 않습니다. 커널 재시작으로 참조가 사라진 서버는 PID를 추측해서 죽이지 않습니다. 포트 충돌 시 IT/서버 터미널에서 실행 기록을 확인하세요.


In [ ]:
STOP_SERVER = False  # 저장 완료 후 True
if STOP_SERVER:
    server = globals().get("TEM_SERVER")
    if server is None:
        print("현재 커널에는 소유한 서버 참조가 없습니다. 다른 프로세스를 종료하지 않습니다.")
    elif server.stop():
        print("웹 서버 종료 완료. 프로젝트와 모델 파일은 보존됩니다.")
else:
    print("서버 종료 생략")


## GPU로 얼마나 빨라지나요?
- **GPU 대상:** SAM 이미지 인코딩, 프롬프트 추론, 학습 Decoder/Refiner.
- **현재 CPU:** EasyOCR(`gpu=False`), Canny/Sobel 등 전처리, Gradient/DP, 회전, 두께/CD 계측.
- 일괄 SAM은 이미지마다 순차 처리합니다. 자동 다중 GPU 분산은 아닙니다.
- GPU 모델/VRAM/점 수/이미지/다른 학습 작업에 따라 달라 **속도 배수는 회사 서버 실측 전 보장하지 않습니다.**
- 현재 소스는 ViT-H decoder를 ViT-B에 억지로 붙이지 않고 기본 모델 SHA256/variant 불일치 시 중단합니다.

참고: [PyTorch 설치 및 CUDA 확인](https://pytorch.org/get-started/locally/), [공식 SAM](https://github.com/facebookresearch/segment-anything), [Jupyter 프록시 설치](https://jupyter-server-proxy.readthedocs.io/en/latest/install.html).

**다음 실행:** 설치 False → 환경/경로 확인 → 필요하면 SAM 시험 생략 → 서버 시작. 매번 재설치/재다운로드할 필요는 없습니다.
